# Módulo 3 · Clase 3 — Búsqueda y procesamiento de texto

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | `grep` | Buscar texto dentro de archivos |
| 2 | `find` | Buscar archivos en el sistema |
| 3 | Pipe `\|` | Encadenar comandos |
| 4 | Redirecciones `>`, `>>`, `<`, `2>` | Capturar y redirigir salidas |
| 5 | `wc`, `sort`, `uniq`, `cut` | Herramientas de procesamiento de texto |
| 6 | Práctica | Analizar logs y datos desde la terminal |

> 💡 **Idea central:** El verdadero poder de la terminal no está en los comandos individuales — está en **combinarlos**. Con pipes y redirecciones, construyes pipelines de procesamiento que hacen en una línea lo que un script largo haría en veinte.

---
## Preparar el entorno de demo

In [ ]:
import os, shutil
from pathlib import Path

DEMO = Path.home() / "busqueda_demo"
if DEMO.exists(): shutil.rmtree(DEMO)
(DEMO / "logs").mkdir(parents=True)
(DEMO / "src" / "modelos").mkdir(parents=True)
(DEMO / "datos").mkdir()
os.chdir(DEMO)

# Logs de entrenamiento
import math, random
random.seed(42)
lineas_log = []
for i in range(1, 31):
    loss = 2.0 * math.exp(-i * 0.08) + random.uniform(-0.02, 0.02)
    acc  = 1 - loss / 2.2
    lineas_log.append(f"[INFO] Epoch {i:02d}/30 loss={loss:.4f} acc={acc:.4f}")
    if i % 5 == 0:
        lineas_log.append(f"[WARNING] val_loss mayor que train_loss en epoch {i}")
    if i == 15:
        lineas_log.append("[ERROR] NaN detectado en gradiente — epoch 15")
Path("logs/entrenamiento.log").write_text("\n".join(lineas_log) + "\n", encoding="utf-8")

# Archivos Python
Path("src/main.py").write_text("""\
import modelo
import preprocesado

def main():
    datos = preprocesado.cargar("datos/train.csv")
    m = modelo.Modelo(capas=[64, 32])
    m.entrenar(datos)
    print("Entrenamiento completado")

if __name__ == "__main__":
    main()
""", encoding="utf-8")

Path("src/modelos/red_neuronal.py").write_text("""\
class RedNeuronal:
    def __init__(self, capas):
        self.capas = capas
        self.pesos = {}

    def entrenar(self, datos, epochs=10, lr=0.001):
        for epoch in range(epochs):
            loss = self._paso_entrenamiento(datos)
            print(f"Epoch {epoch}: loss={loss:.4f}")

    def predecir(self, x):
        return x  # placeholder

    def _paso_entrenamiento(self, datos):
        return 0.5  # placeholder
""", encoding="utf-8")

# CSV con datos
Path("datos/ventas.csv").write_text(
    "mes,producto,ventas,region\n" +
    "\n".join([
        f"{mes},{prod},{v},{'Norte' if i%2==0 else 'Sur'}"
        for i, (mes, prod, v) in enumerate([
            ("Ene","A",100),("Ene","B",200),("Feb","A",150),
            ("Feb","B",250),("Mar","A",175),("Mar","B",300),
            ("Abr","A",200),("Abr","B",350),("May","A",225),
            ("May","B",400)
        ])
    ]) + "\n",
    encoding="utf-8"
)

print(f"Demo en: {DEMO}")
!ls -R

---
## 1. `grep` — Buscar texto en archivos

`grep` (*global regular expression print*) busca líneas que coincidan con un patrón.

```bash
grep "patrón" archivo          # buscar en un archivo
grep -i "patrón" archivo       # ignorar mayúsculas/minúsculas
grep -n "patrón" archivo       # mostrar número de línea
grep -r "patrón" directorio/   # buscar recursivamente
grep -l "patrón" directorio/   # solo nombres de archivos con coincidencias
grep -v "patrón" archivo       # líneas que NO coinciden (invertir)
grep -c "patrón" archivo       # contar coincidencias
grep -E "patrón" archivo       # expresiones regulares extendidas
grep -A 2 "patrón" archivo     # 2 líneas después de cada coincidencia
grep -B 2 "patrón" archivo     # 2 líneas antes
grep -C 2 "patrón" archivo     # 2 líneas de contexto (antes y después)
```

In [ ]:
# Buscar errores en el log
print("=== Líneas con ERROR ===")
!grep "ERROR" logs/entrenamiento.log

print("\n=== Líneas con WARNING (con número de línea) ===")
!grep -n "WARNING" logs/entrenamiento.log

In [ ]:
# Contar coincidencias
print("=== Cuántas líneas INFO hay ===")
!grep -c "INFO" logs/entrenamiento.log

# Invertir: líneas que NO son INFO
print("\n=== Líneas que NO son INFO ===")
!grep -v "\[INFO\]" logs/entrenamiento.log

# Buscar con contexto
print("\n=== ERROR con 2 líneas de contexto ===")
!grep -C 2 "ERROR" logs/entrenamiento.log

In [ ]:
# grep -E: expresiones regulares extendidas
print("=== Epochs donde acc > 0.8 (regex) ===")
!grep -E 'acc=0\.[89][0-9]+' logs/entrenamiento.log

# Buscar en código fuente
print("\n=== Todas las funciones def en el código ===")
!grep -rn "def " src/

print("\n=== Archivos que importan algo ===")
!grep -rl "^import " src/

---
## 2. `find` — Buscar archivos en el sistema

Mientras `grep` busca **dentro** de archivos, `find` busca **archivos** según sus propiedades.

```bash
find . -name "*.py"              # archivos .py en el directorio actual
find . -name "*.py" -type f      # asegurar que son archivos (no dirs)
find . -type d                   # solo directorios
find . -mtime -7                 # modificados en los últimos 7 días
find . -mtime +30                # modificados hace más de 30 días
find . -size +1M                 # archivos mayores de 1 MB
find . -size -100k               # archivos menores de 100 KB
find . -name "*.tmp" -delete     # buscar y borrar en un paso
find . -name "*.py" -exec wc -l {} \;  # ejecutar comando en cada resultado
find . -empty                    # archivos o directorios vacíos
find . -perm 644                 # archivos con permisos exactos 644
```

In [ ]:
# Buscar todos los .py
print("=== Archivos .py ===")
!find . -name "*.py" -type f

print("\n=== Directorios ===")
!find . -type d

print("\n=== Archivos mayores de 500 bytes ===")
!find . -type f -size +500c   # c = bytes

In [ ]:
# -exec: ejecutar un comando en cada resultado
print("=== Líneas en cada .py ===")
!find . -name "*.py" -exec wc -l {} \;

# Crear algunos archivos temporales para practicar -delete
!touch temp1.tmp temp2.tmp logs/debug.tmp
print("\n=== .tmp antes de limpiar ===")
!find . -name "*.tmp"

!find . -name "*.tmp" -delete
print("\n=== .tmp después de limpiar ===")
!find . -name "*.tmp" || echo "(ninguno)"

---
## 3. Pipe `|` — Encadenar comandos

El **pipe** (`|`) conecta la salida de un comando con la entrada del siguiente. Es el mecanismo más poderoso de la terminal.

```
comando1 | comando2 | comando3

La salida de comando1 alimenta comando2,
cuya salida alimenta comando3.
```

```bash
# Ejemplos progresivos:
cat log.txt | grep ERROR               # filtrar errores
cat log.txt | grep ERROR | wc -l       # contar errores
ls -la | grep ".py"                    # listar solo .py
cat datos.csv | cut -d',' -f3 | sort   # extraer columna y ordenar
ps aux | grep python                   # procesos Python activos
find . -name "*.py" | xargs wc -l      # líneas de todos los .py
```

In [ ]:
# Pipeline simple: filtrar y contar
print("=== ¿Cuántos errores/warnings hay? ===")
!cat logs/entrenamiento.log | grep -E "ERROR|WARNING" | wc -l

# Pipeline intermedio: top 5 de algo
print("\n=== Las 5 funciones más largas (por líneas) ===")
!find . -name "*.py" | xargs grep -c "def " 2>/dev/null | sort -t: -k2 -rn | head -5

# Contar palabras únicas en logs
print("\n=== Niveles de log únicos ===")
!cat logs/entrenamiento.log | grep -oE '\[\w+\]' | sort | uniq -c | sort -rn

In [ ]:
# Pipeline con cut: extraer columnas de CSV
print("=== Solo la columna 'ventas' del CSV ===")
!cat datos/ventas.csv | cut -d',' -f3

print("\n=== Ventas ordenadas de mayor a menor ===")
!cat datos/ventas.csv | tail -n +2 | cut -d',' -f3 | sort -rn | head -5

print("\n=== Regiones únicas ===")
!cat datos/ventas.csv | cut -d',' -f4 | sort | uniq

---
## 4. Redirecciones — Capturar y redirigir

```bash
comando > archivo.txt       # redirigir stdout a archivo (sobreescribe)
comando >> archivo.txt      # añadir al final del archivo (append)
comando < archivo.txt       # leer stdin desde archivo
comando 2> errores.txt      # redirigir stderr a archivo
comando 2>&1                # redirigir stderr al mismo sitio que stdout
comando > todo.txt 2>&1     # capturar tanto stdout como stderr
comando > /dev/null         # descartar la salida
comando 2> /dev/null        # suprimir errores
```

```
CANALES ESTÁNDAR:
  0 → stdin  (entrada estándar)
  1 → stdout (salida estándar)  ← > y >>
  2 → stderr (errores)          ← 2>
```

In [ ]:
# > sobreescribe, >> añade
!echo "Primera línea" > resultado.txt
!echo "Segunda línea" >> resultado.txt
!echo "Tercera línea" >> resultado.txt
print("Contenido de resultado.txt:")
!cat resultado.txt

# Capturar salida de un pipeline
!grep "ERROR" logs/entrenamiento.log > errores_filtrados.txt
print("\nErrores capturados:")
!cat errores_filtrados.txt

In [ ]:
# 2> para capturar errores (stderr)
!ls archivo_inexistente.txt 2> error_ls.txt
print("Error capturado en error_ls.txt:")
!cat error_ls.txt

# Suprimir errores con /dev/null
print("\nBuscando *.xyz (error silenciado):")
!find . -name "*.xyz" 2>/dev/null || echo "(sin resultados)"

# Capturar TODO (stdout + stderr) en el mismo archivo
!find . -type f > todo_busqueda.txt 2>&1
print("\nTodo capturado:")
!cat todo_busqueda.txt

---
## 5. Herramientas de procesamiento de texto

```bash
wc -l archivo          # contar líneas
wc -w archivo          # contar palabras
wc -c archivo          # contar bytes

sort archivo           # ordenar líneas alfabéticamente
sort -n archivo        # ordenar numéricamente
sort -r archivo        # orden inverso
sort -k2 archivo       # ordenar por campo 2
sort -t',' -k3 -n      # CSV: ordenar por columna 3 numéricamente

uniq archivo           # eliminar líneas duplicadas consecutivas*
uniq -c archivo        # contar repeticiones
uniq -d archivo        # solo las duplicadas

cut -d',' -f1,3        # extraer columnas 1 y 3 (delimitador ,)
cut -c1-10             # extraer caracteres 1 a 10

# * uniq solo elimina duplicados CONSECUTIVOS → siempre sort antes
sort archivo | uniq    # patrón correcto para eliminar duplicados
```

In [ ]:
# sort con CSV
print("=== ventas.csv ordenado por ventas (columna 3, numérico) ===")
!head -n 1 datos/ventas.csv    # cabecera
!tail -n +2 datos/ventas.csv | sort -t',' -k3 -rn

print("\n=== Productos únicos ===")
!cut -d',' -f2 datos/ventas.csv | sort | uniq

print("\n=== Conteo de apariciones por mes ===")
!cut -d',' -f1 datos/ventas.csv | tail -n +2 | sort | uniq -c

In [ ]:
# Pipeline complejo: análisis de log en una línea
print("=== Top 3 epochs con mayor acc (desde el log) ===")
!grep '\[INFO\]' logs/entrenamiento.log \
    | grep -oE 'acc=[0-9.]+' \
    | cut -d= -f2 \
    | sort -rn \
    | head -3

print("\n=== Resumen: cuántas líneas por nivel ===")
!cat logs/entrenamiento.log \
    | grep -oE '\[(INFO|WARNING|ERROR)\]' \
    | sort | uniq -c | sort -rn

---
## 6. Práctica — Analizar logs y datos

---

### Ejercicio 1 — Buscar en el código fuente

**Enunciado:** Usando `grep` y `find` (sin Python), responde a estas preguntas sobre el directorio `~/busqueda_demo/src/`:

1. ¿Cuántas funciones (`def`) hay en total?
2. ¿Qué archivos importan algún módulo?
3. ¿En qué línea aparece la palabra `epochs`?
4. ¿Cuántos archivos `.py` hay en total?
5. Muestra todas las clases definidas (`class`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import os
from pathlib import Path
os.chdir(Path.home() / "busqueda_demo")


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```bash
# 1. Total de funciones def
grep -r "def " src/ | wc -l

# 2. Archivos que importan módulos
grep -rl "^import \|^from " src/

# 3. Línea donde aparece 'epochs'
grep -rn "epochs" src/

# 4. Cantidad de .py
find src/ -name "*.py" -type f | wc -l

# 5. Clases definidas
grep -rn "^class " src/
```
</details>

---
### Ejercicio 2 — Pipelines de análisis

**Enunciado:** Construye cada pipeline en una sola línea de terminal:

1. Muestra las 5 líneas del log con el valor de `loss` más alto.
2. Cuenta cuántas épocas tuvieron accuracy superior a 0.75.
3. Extrae solo los valores de `acc` del log, conviértelos a lista ordenada de mayor a menor.
4. Del CSV de ventas, muestra el mes con mayor total de ventas (suma por mes).
5. Muestra los archivos modificados en las últimas 24 horas (usa `find -mtime`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```bash
# 1. Top 5 líneas con mayor loss
grep -oE 'loss=[0-9.]+' logs/entrenamiento.log | \
  cut -d= -f2 | sort -rn | head -5

# 2. Épocas con acc > 0.75
grep -E 'acc=0\.[789][0-9]*' logs/entrenamiento.log | wc -l

# 3. Valores de acc ordenados
grep -oE 'acc=[0-9.]+' logs/entrenamiento.log | \
  cut -d= -f2 | sort -rn

# 4. Mes con mayor total de ventas
# (requiere awk, que vemos brevemente aquí)
tail -n +2 datos/ventas.csv | \
  awk -F',' '{sum[$1]+=$3} END {for(m in sum) print sum[m], m}' | \
  sort -rn | head -1

# 5. Archivos modificados en las últimas 24 horas
find . -type f -mtime -1
```

**Nota sobre `awk`:** Es una herramienta de procesamiento de texto muy potente que merece un módulo propio. En este curso lo usamos puntualmente para operaciones aritméticas en columnas que `cut` no puede hacer.
</details>

---
### Ejercicio 3 — Redirecciones

**Enunciado:** Practica las redirecciones:

1. Guarda en `informe_errores.txt` todas las líneas de error y warning del log.
2. Ejecuta `ls directorio_que_no_existe` y captura el error en `error.txt`.
3. Ejecuta un `find` que mezcle resultados normales y errores; guarda TODO en `busqueda_completa.txt`.
4. Añade una línea de resumen al final de `informe_errores.txt` con la fecha actual: `Generado: YYYY-MM-DD`.
5. Verifica el contenido final de `informe_errores.txt`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```bash
# 1. Guardar errores y warnings
grep -E 'ERROR|WARNING' logs/entrenamiento.log > informe_errores.txt

# 2. Capturar error de ls
ls directorio_que_no_existe 2> error.txt
cat error.txt

# 3. find con stdout + stderr
find / -name "*.py" -maxdepth 5 > busqueda_completa.txt 2>&1
wc -l busqueda_completa.txt

# 4. Añadir línea de fecha
echo "Generado: $(date +%Y-%m-%d)" >> informe_errores.txt

# 5. Verificar
cat informe_errores.txt
```

**`$(...)` es la sustitución de comandos:** el resultado del comando interior se inserta como texto. `$(date +%Y-%m-%d)` produce la fecha actual formateada. Lo vemos en detalle en la Clase 4 (Bash scripting).
</details>

---
### Ejercicio 4 — Búsqueda avanzada con find

**Enunciado:** Usando solo `find` (con `-exec` o `-delete` donde corresponda):

1. Encuentra todos los archivos vacíos en `~/busqueda_demo/`.
2. Encuentra todos los directorios vacíos.
3. Encuentra archivos con permisos de escritura para «otros» (peligrosos) con `-perm /o+w`.
4. Busca archivos `.log` modificados en los últimos dos días.
5. Lista todos los archivos con su tamaño usando `-exec ls -lh {} \;`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```bash
# 1. Archivos vacíos
find ~/busqueda_demo -type f -empty

# 2. Directorios vacíos
find ~/busqueda_demo -type d -empty

# 3. Archivos con o+w (escritura para otros)
find ~/busqueda_demo -type f -perm /o+w

# 4. Logs modificados en los últimos 2 días
find ~/busqueda_demo -name "*.log" -mtime -2

# 5. Todos los archivos con tamaño
find ~/busqueda_demo -type f -exec ls -lh {} \;
```
</details>

---
### Ejercicio 5 (reto) — Analizador de logs con Python + subprocess

**Enunciado:** Escribe la función `analizar_logs(directorio)` que use `subprocess` para ejecutar pipelines de shell y genere un informe completo del directorio de logs:

- Total de líneas en todos los `.log`.
- Desglose por nivel (INFO / WARNING / ERROR) con porcentajes.
- Los tres valores de `loss` más altos encontrados en todos los logs.
- Archivos de log con errores (nombre + número de errores).
- Guardar el informe en `informe_logs.txt`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import subprocess
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import subprocess, os
from pathlib import Path
from datetime import datetime

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    return r.stdout.strip()

def analizar_logs(directorio):
    d = Path(directorio).expanduser().resolve()

    logs = list(d.glob("**/*.log"))
    if not logs:
        print("No se encontraron archivos .log")
        return

    # Total de líneas
    total_lineas = int(sh(f"cat {' '.join(str(l) for l in logs)} | wc -l"))

    # Desglose por nivel
    niveles = {}
    for nivel in ("INFO", "WARNING", "ERROR"):
        count = sh(f"cat {' '.join(str(l) for l in logs)} | grep -c '{nivel}' || echo 0")
        niveles[nivel] = int(count)

    # Top 3 valores de loss
    top_loss_raw = sh(
        f"cat {' '.join(str(l) for l in logs)} | "
        f"grep -oE 'loss=[0-9.]+' | cut -d= -f2 | sort -rn | head -3"
    )

    # Archivos con errores
    errores_por_archivo = {}
    for log in logs:
        n = sh(f"grep -c 'ERROR' {log} || echo 0")
        if int(n) > 0:
            errores_por_archivo[log.name] = int(n)

    # Construir informe
    lineas = [
        f"INFORME DE LOGS — {d}",
        f"Generado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Archivos analizados: {len(logs)}",
        f"Total de líneas: {total_lineas}",
        "",
        "DESGLOSE POR NIVEL:",
    ]
    for nivel, count in niveles.items():
        pct = (count / total_lineas * 100) if total_lineas else 0
        lineas.append(f"  {nivel:<10} {count:>4}  ({pct:.1f}%)")

    lineas += ["", "TOP 3 VALORES DE LOSS:"] + [
        f"  {v}" for v in top_loss_raw.splitlines()
    ]

    lineas += ["", "ARCHIVOS CON ERRORES:"]
    if errores_por_archivo:
        for nombre, n in sorted(errores_por_archivo.items()):
            lineas.append(f"  {nombre}: {n} error(es)")
    else:
        lineas.append("  Ninguno")

    informe = "\n".join(lineas)
    print(informe)

    Path(d / "informe_logs.txt").write_text(informe + "\n", encoding="utf-8")
    print(f"\n✅ Informe guardado en {d}/informe_logs.txt")


analizar_logs("~/busqueda_demo/logs")
```
</details>

---
## Resumen de la clase

```bash
# ─── grep ──────────────────────────────────────────────────
grep "patrón" archivo       # buscar en archivo
grep -i "patrón" archivo    # ignorar mayúsculas
grep -n "patrón" archivo    # con número de línea
grep -r "patrón" dir/       # recursivo
grep -v "patrón" archivo    # invertir
grep -c "patrón" archivo    # contar
grep -E "regex" archivo     # expresiones regulares
grep -C 2 "patrón" archivo  # 2 líneas de contexto

# ─── find ──────────────────────────────────────────────────
find . -name "*.py"         # por nombre
find . -type f/d            # archivos / directorios
find . -mtime -7            # modificados hace < 7 días
find . -size +1M            # mayores de 1 MB
find . -empty               # vacíos
find . -name "*.tmp" -delete  # buscar y borrar
find . -name "*.py" -exec wc -l {} \;  # ejecutar comando

# ─── Pipes y redirecciones ─────────────────────────────────
cmd1 | cmd2                 # pipe
cmd > archivo               # stdout → archivo (sobreescribir)
cmd >> archivo              # stdout → archivo (añadir)
cmd 2> errores              # stderr → archivo
cmd > todo.txt 2>&1         # stdout + stderr → archivo
cmd 2>/dev/null             # suprimir errores

# ─── Herramientas de texto ─────────────────────────────────
wc -l archivo               # contar líneas
sort -n archivo             # ordenar numéricamente
sort | uniq                 # eliminar duplicados
uniq -c                     # contar repeticiones
cut -d',' -f3               # extraer columna CSV
```

## ¿Qué viene en la próxima clase?

En la **Clase 4** aprendemos **Bash scripting**: variables, condicionales, bucles y funciones. Con eso tendremos las piezas para el proyecto final de automatización.